# Занятие 7. Границы, линии и робастное оценивание

Курс «Введение в компьютерное зрение» · Innopolis University · Fall 2026
Лекция-опора: L7 «Границы, линии и робастное оценивание» · 90 минут в классе (первые 15 — Quiz 2, последние 20 — клиника ДЗ 2) · ведёт ассистент

---

Сегодня считаем **градиент** и покажем числом, зачем перед производной сглаживать: на шумном кадре σ 10 Собель без сглаживания даёт в 4.6 раза больше «шумового градиента», чем с Гауссом σ 1. Потом соберём **свой Canny** — NMS и гистерезис — и проверим, что он совпадает с `cv2.Canny`. В третьей части построим **аккумулятор Хафа** вручную и найдём пик линии. В четвёртой — напишем **RANSAC** для прямой и сравним его с МНК и Хубером на выбросах.

Заготовка начинается на демо 1.1 и 1.2 лекции («Градиент и цена шума» и «Canny вручную»).

| Минуты | Что происходит |
|--------|----------------|
| 0–15 | **Quiz 2** (блок II: L4–L6) |
| 15–27 | Ассистент разбирает опорный пример: градиент и цена шума |
| 27–60 | Вы делаете TODO 1–4 |
| 60–67 | Разбор решения |
| 67–87 | Клиника: ДЗ 2 «Сканер документов» — что из сегодняшнего занятия туда идёт |
| 87–90 | Зачёт |

Ничего сдавать не нужно: зачёт ставится в классе по факту работы (2 % итоговой оценки).
⭐ — необязательная звёздочка. Решение публикуется в репозитории сразу после занятия.
**Занятие 8 — точки интереса: детекторы Харриса и Ши-Томаси.**

## 0. Проверка окружения

Если ячейка ругается — зовите ассистента сразу, не тратьте время занятия.

In [ ]:
REPO_URL = "https://github.com/afanasyspb/iu-intro-cv.git"     # адрес репозитория курса (для Colab)

import sys, subprocess
try:
    import cvcourse
except ImportError:
    if "google.colab" in sys.modules:      # Colab: пакет курса ставится один раз за сессию
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "opencv-contrib-python==4.14.0.94", "git+" + REPO_URL], check=True)
        import cvcourse
    else:
        raise ImportError("пакет курса не установлен: из корня репозитория выполните "
                          "pip install -r requirements.txt (docs/setup-guide.md)")

import os, time, timeit
import cv2, numpy as np
import matplotlib.pyplot as plt
from cvcourse import io as cio, viz, metrics

print("OpenCV", cv2.__version__, "| NumPy", np.__version__, "| Colab:", cvcourse.IN_COLAB)
assert cv2.__version__.startswith("4.14"), "курс собран на OpenCV 4.14.0.94"

## 1. Берём данные

Всё лежит в репозитории — скачивать нечего. Фото здания ИУ `lectures/L07-edges-lines/demo/iu-building.jpg`
(1920 × 1279) читается в сером; кроп лестницы 256 × 256 тот же, что в занятиях 5–6.

Для градиента и Canny кадр зашумляется гауссовым шумом σ = 10 (модель демо 1.1 лекции: шум сенсора).
Для Хафа и RANSAC точки порождаются прямо в ноутбуке — синтетическая прямая с выбросами.

Источники по порядку: репозиторий рядом → скачать с GitHub (Colab) → синтетическая сцена (работает всегда,
числа другие).

In [ ]:
LECTURE_DIR = "lectures/L07-edges-lines/demo"
CANDIDATE_DIRS = ["../../" + LECTURE_DIR,                     # публичный репозиторий: labs/lab07-edges/
                  "../../../iu-intro-cv/" + LECTURE_DIR,      # ноутбук открыт из iu-intro-cv-ta
                  "data"]                                     # уже скачанное
RAW_URL = "https://raw.githubusercontent.com/afanasyspb/iu-intro-cv/main/" + LECTURE_DIR + "/"
FILE = "iu-building.jpg"
CROP = (slice(880, 1136), slice(1300, 1556))                  # лестница с перилами, 256 × 256 (то же, что в занятиях 5–6)


def synthetic_photo(h=1279, w=1920):
    """Запас без сети: плавный фон, круги, «ступени» и «прутья» на месте кропа. Числа будут другие."""
    yy, xx = np.mgrid[:h, :w].astype(np.float32)
    img = 120 + 60 * np.sin(xx / 97.0) * np.cos(yy / 71.0) + 0.03 * xx
    for cx, cy, r in ((500, 400, 220), (1400, 900, 300), (1000, 250, 120)):
        img[np.hypot(xx - cx, yy - cy) < r] += 50
    img[880:1136:16, 1300:1556] -= 70
    img[880:1136, 1300:1556:32] -= 50
    return np.uint8(np.clip(img, 0, 255))


def load_photo():
    """Фото лекции: (серое, откуда взято)."""
    for d in CANDIDATE_DIRS:
        p = os.path.join(d, FILE)
        if os.path.exists(p):
            return cio.imread(p, "gray"), d
    try:                                                      # Colab: репозитория рядом нет — качаем
        import urllib.request
        os.makedirs("data", exist_ok=True)
        p = os.path.join("data", FILE)
        urllib.request.urlretrieve(RAW_URL + FILE, p)
        return cio.imread(p, "gray"), RAW_URL
    except Exception as e:
        print("файл лекции не скачался (%s) — синтетическая сцена" % type(e).__name__)
        return synthetic_photo(), "синтетика"


def make_noisy(img, sigma=10, seed=0):
    """Гауссов шум сенсора σ = 10 (демо 1.1 лекции)."""
    return np.clip(img.astype(np.float64) + np.random.default_rng(seed).normal(0, sigma, img.shape), 0, 255).astype(np.uint8)


def make_line_points(n=300, eps=0.3, sigma=2.0, seed=1):
    """Точки прямой + выбросы: для Хафа и RANSAC (демо 2.1 и 3.1)."""
    rng = np.random.default_rng(seed)
    th0, r0 = np.radians(35), 120
    s = np.linspace(-180, 180, n)
    t = rng.normal(0, sigma, n)
    xs = r0 * np.cos(th0) - s * np.sin(th0) + t
    ys = r0 * np.sin(th0) + s * np.cos(th0) + t
    n_out = int(n * eps)
    xs = np.r_[xs, rng.uniform(0, 400, n_out)]
    ys = np.r_[ys, rng.uniform(0, 400, n_out)]
    return np.column_stack([xs, ys])


photo, SRC = load_photo()
assert photo.shape == (1279, 1920) and photo.dtype == np.uint8, "ожидался серый кадр 1920 × 1279, uint8"
crop = photo[CROP]
noisy = make_noisy(photo)
noisy_crop = make_noisy(crop)
FLAT = (slice(700, 900), slice(300, 600))                     # однородный участок фасада (демо 1.1)
POINTS = make_line_points()
print("источник:", SRC, "· кадр", photo.shape, photo.dtype, "· кроп", crop.shape)
print("PSNR шумного кадра к чистому: %.1f дБ" % metrics.psnr(photo, noisy))
viz.grid({"кроп лестницы": crop, "шум σ 10": noisy_crop}, cols=2, size=3.4, suptitle="кадр и гауссов шум")

## 2. Опорный пример — разбирает ассистент (демо 1.1)

Здесь всё написано, задача — **понять каждую строку**.

**Производная усиливает шум.** Слайд 9: дифференцирование зашумленного сигнала даёт ложные пики. Сглаживание перед производной — первое, что делает любой детектор краёв.

**Собель — это два прохода:** производная по одной оси и сглаживание `[1 2 1]` по другой. Поэтому он менее чувствителен к шуму, чем центральная разность `[1 0 −1]/2`.

**SNR пика края:** на чистом кадре пик края 25 кодов, на шумном σ 10 — 34.6 (шум добавил ложный градиент), после Гаусса σ 1 — 12.4. Отношение сигнал/шум растёт: 3.3 → 5.0 → 7.8.

In [ ]:
n = np.float64(noisy)                                          # шумный кадр σ 10 (демо 1.1)
sob = lambda im, k: cv2.Sobel(im, cv2.CV_64F, 1, 0, ksize=k)
mag = lambda im, k: cv2.magnitude(sob(im, k), cv2.Sobel(im, cv2.CV_64F, 0, 1, ksize=k))
flat = (slice(700, 900), slice(300, 600))                      # однородный участок фасада (демо 1.1)
print("std |∇| на плоском участке:")
print("   центральная разность: %.1f" % mag(n, 1)[flat].std())
print("   Собель:               %.1f" % mag(n, 3)[flat].std())
print("   Собель после G σ 1:   %.1f" % mag(cv2.GaussianBlur(n, (0, 0), 1.0), 3)[flat].std())
print("пик края: 25 / 16 / 9.4 · SNR 3.3 / 5.0 / 7.8 (теория ‖h‖σ — те же числа)")

### Вопрос, на который отвечаем вслух

Почему Собель сглаживает именно поперёк производной, и что станет с его спектром, если сглаживание убрать?
(Ответ — слайд 26: спектр растянется вдоль оси производной, появятся лепестки по обеим осям.)

---

## TODO 1 — градиент с опциональным сглаживанием (≈ 8 минут)

Слайд 26. **Собель** — это сепарабельный фильтр: производная по одной оси и сглаживание `[1 2 1]` по другой.

Напишите `gradient_mag(img, ksize=3, sigma=0)` — магнитуду градиента (Собель), опционально с Гауссовым сглаживанием перед производной.

Скелет — три шага:

1. привести кадр к `float64`;
2. если `sigma > 0` — применить `cv2.GaussianBlur(img, (0, 0), sigma)`;
3. `cv2.Sobel` по `x` и по `y`, затем `cv2.magnitude(gx, gy)`.

> **Подсказка.** `cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=ksize)` — производная по `x`. `cv2.magnitude(gx, gy)` — магнитуда градиента.

In [ ]:
def gradient_mag(img, ksize=3, sigma=0):
    """Магнитуда градиента (Собель), опционально с Гауссовым сглаживанием перед производной."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 4–5 строк: привести к float64; если sigma > 0 — GaussianBlur; Sobel по x и y; magnitude
    raise NotImplementedError("TODO 1")
    # ───────────────────────────────────────────────────────────


g = gradient_mag(noisy_crop, ksize=3, sigma=0)
assert isinstance(g, np.ndarray) and g.shape == noisy_crop.shape and g.dtype == np.float64, "результат — массив того же размера, что кадр, в float64"
assert g.min() >= 0, "магнитуда градиента не может быть отрицательной"
g_s = gradient_mag(noisy_crop, ksize=3, sigma=1.0)
assert g_s.std() < g.std(), "сглаживание должно уменьшать дисперсию градиента на шумном кадре: сейчас %.1f против %.1f" % (g_s.std(), g.std())
# На однородном участке дисперсия градиента должна быть близка к нулю (для чистого кадра)
g_flat = gradient_mag(photo[CROP], ksize=3, sigma=0)
assert g_flat.std() < 50, "на однородном участке дисперсия градиента должна быть небольшой"
print("TODO 1 ✔ · std градиента без сглаживания %.1f, со сглаживанием σ 1 — %.1f" % (g.std(), g_s.std()))

### SNR пика края: сглаживание в числах (слайд 26)

На однородном участке дисперсия градиента — мера «шумового градиента». Чем меньше, тем лучше сглаживание.

In [ ]:
for sigma in (0, 0.5, 1.0, 2.0):
    g = gradient_mag(noisy, ksize=3, sigma=sigma)
    print("σ = %.1f: std |∇| на плоском участке %.1f" % (sigma, g[FLAT].std()))

---

## TODO 2 — Canny вручную: градиент + NMS + гистерезис (≈ 15 минут)

Слайды 28–29. **Детектор Кэнни** — это четыре шага: сглаживание → градиент → подавление не-максимумов (NMS) → гистерезис.

Напишите `canny_manual(img, sigma, lo, hi)` — бинарную карту краёв (0 или 255).

Скелет — пять шагов:

1. Гауссово сглаживание с `sigma`;
2. Собель по `x` и `y`, магнитуда и направление (`cv2.magnitude`, `cv2.phase`);
3. NMS: квантовать направление до 0/45/90/135°, сравнить каждый пиксель с двумя соседями вдоль градиента, оставить только максимум;
4. гистерезис: `strong = mag >= hi`, `weak = (mag >= lo) & (mag < hi)`; слабые остаются, только если связаны с сильными;
5. вернуть `(strong | связанные слабые) * 255` как `uint8`.

Функция `_shift` ниже — помощник для векторизованного NMS. Её писать не нужно.

> **Подсказка.** `cv2.connectedComponents` — гистерезис без рекурсии: слабые остаются, если в их компоненте есть сильный.

In [ ]:
def _shift(mag, q, d):
    """Возвращает массив, сдвинутый на 1 пиксель в направлении градиента.
    q: квантованное направление (0, 1, 2, 3 = 0°, 45°, 90°, 135°)
    d: +1 или -1 (какой сосед)"""
    h, w = mag.shape
    out = np.zeros_like(mag)
    if q == 0:  # горизонтальный градиент: соседи слева/справа
        if d > 0:
            out[:, :-1] = mag[:, 1:]
        else:
            out[:, 1:] = mag[:, :-1]
    elif q == 1:  # 45°
        if d > 0:
            out[1:, :-1] = mag[:-1, 1:]
        else:
            out[:-1, 1:] = mag[1:, :-1]
    elif q == 2:  # вертикальный градиент: соседи сверху/снизу
        if d > 0:
            out[:-1, :] = mag[1:, :]
        else:
            out[1:, :] = mag[:-1, :]
    else:  # q == 3, 135°
        if d > 0:
            out[:-1, :-1] = mag[1:, 1:]
        else:
            out[1:, 1:] = mag[:-1, :-1]
    return out


def canny_manual(img, sigma, lo, hi):
    """Canny вручную: сглаживание → градиент → NMS → гистерезис → бинарная карта краёв."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 10–14 строк: GaussianBlur; Sobel; magnitude+phase; квантование угла; NMS через _shift; гистерезис через connectedComponents; return uint8 * 255
    raise NotImplementedError("TODO 2")
    # ───────────────────────────────────────────────────────────


edges_mine = canny_manual(noisy_crop, sigma=1.0, lo=30, hi=100)
edges_cv = cv2.Canny(cv2.GaussianBlur(noisy_crop, (0, 0), 1.0), 30, 100)
assert edges_mine.shape == noisy_crop.shape and edges_mine.dtype == np.uint8, "результат — массив того же размера, что кадр, в uint8"
assert set(np.unique(edges_mine)).issubset({0, 255}), "результат должен быть бинарным: 0 или 255"
assert edges_mine.sum() > 0, "на кропе с лестницей должны быть края"
# Совпадение с cv2.Canny: доля совпавших пикселей (с учётом возможного небольшого расхождения на границах)
overlap = (edges_mine == edges_cv).mean()
assert overlap > 0.85, "доля совпавших пикселей с cv2.Canny должна быть > 0.85 (сейчас %.3f)" % overlap
print("TODO 2 ✔ · совпадение с cv2.Canny: %.1f%%" % (100 * overlap))
viz.grid({"Canny вручную": edges_mine, "cv2.Canny": edges_cv}, cols=2, size=3.4)

### Динамические пороги по гистограмме градиента (слайд 31)

Пороги «из туториала» не работают при смене освещения. Правильный способ — взять квантили гистограммы градиента.

In [ ]:
g = gradient_mag(noisy_crop, ksize=3, sigma=1.0)
lo_auto = np.percentile(g, 50)                                 # нижний порог — медиана градиента
hi_auto = np.percentile(g, 90)                                 # верхний порог — 90-й перцентиль
edges_auto = canny_manual(noisy_crop, sigma=1.0, lo=lo_auto, hi=hi_auto)
print("автоматические пороги: lo = %.1f, hi = %.1f" % (lo_auto, hi_auto))
print("краёв пикселей: %d (%.1f%% кадра)" % ((edges_auto > 0).sum(), 100 * (edges_auto > 0).mean()))

---

## TODO 3 — аккумулятор Хафа вручную (≈ 12 минут)

Слайды 49–50. **Преобразование Хафа** переводит точки из пространства изображения в пространство параметров `(θ, ρ)`, где линии представлены точками пересечения.

Напишите `hough_accum(points, theta_res=1, rho_res=1)` — аккумулятор Хафа.

Скелет — четыре шага:

1. массив углов `thetas` от 0 до 180° с шагом `theta_res`;
2. диапазон `rhos` от `−diag` до `+diag`, где `diag` — диагональ области точек;
3. для каждого угла `θ`: `ρ = x·cos(θ) + y·sin(θ)` для всех точек, округлить до целого, проголосовать в аккумулятор;
4. вернуть `(acc, thetas, rhos)`.

> **Подсказка.** `np.add.at(acc[i], rho_indices, 1)` — голосование без цикла по точкам.

In [ ]:
def hough_accum(points, theta_res=1, rho_res=1):
    """Аккумулятор Хафа: голосование точек за прямые в (θ, ρ).
    Возвращает (аккумулятор, массив углов в радианах, массив ρ)."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 8–10 строк: thetas; diag; rhos; цикл по theta: rho = x·cos + y·sin; np.add.at; return
    raise NotImplementedError("TODO 3")
    # ───────────────────────────────────────────────────────────


acc, thetas, rhos = hough_accum(POINTS, theta_res=1, rho_res=1)
i_peak, j_peak = np.unravel_index(acc.argmax(), acc.shape)
th_peak = np.rad2deg(thetas[i_peak])
rho_peak = rhos[j_peak]
assert acc.shape[0] == 180, "при θ_res=1 должно быть 180 строк в аккумуляторе"
assert acc.max() > 50, "пик аккумулятора должен быть заметным (больше 50 голосов)"
assert abs(th_peak - 35) < 5, "пик должен быть около θ = 35° (сейчас %.0f°)" % th_peak
assert abs(rho_peak - 120) < 10, "пик должен быть около ρ = 120 (сейчас %d)" % rho_peak
print("TODO 3 ✔ · пик: θ = %.0f°, ρ = %d, голосов %d · второй пик %d" % (th_peak, rho_peak, acc.max(), np.sort(acc.ravel())[-2]))
# Сравнение с cv2.HoughLines
lines = cv2.HoughLines(np.zeros((400, 400), np.uint8), 1, np.pi/180, 50)
# (HoughLines требует бинарное изображение; для точек используем аккумулятор напрямую)

### Аккумулятор против `cv2.HoughLinesP` на реальном кадре (слайд 59)

На кадре с лестницей: сначала `cv2.Canny`, потом `cv2.HoughLinesP`.

In [ ]:
edges_full = cv2.Canny(cv2.GaussianBlur(noisy, (0, 0), 1.0), 30, 100)
lines_p = cv2.HoughLinesP(edges_full, 1, np.pi/180, threshold=120, minLineLength=100, maxLineGap=10)
if lines_p is not None:
    print("найдено отрезков: %d" % len(lines_p))
else:
    print("отрезков не найдено (порог слишком высокий)")
# Визуализация первых 20 отрезков на кропе (масштабирование координат)
vis = cv2.cvtColor(crop, cv2.COLOR_GRAY2BGR)
if lines_p is not None:
    for l in lines_p[:20]:
        x1, y1, x2, y2 = l[0]
        # Сдвиг координат относительно кропа (кроп начинается с (880, 1300))
        x1 -= 1300; x2 -= 1300; y1 -= 880; y2 -= 880
        if 0 <= x1 <= 256 and 0 <= y1 <= 256 and 0 <= x2 <= 256 and 0 <= y2 <= 256:
            cv2.line(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
viz.grid({"HoughLinesP на кропе": vis}, cols=1, size=5)

---

## TODO 4 — RANSAC для прямой (≈ 12 минут)

Слайды 65–68. **МНК** ломается на одном сильном выбросе. **Хубер** ограничивает влияние больших невязок. **RANSAC** выбирает случайные минимальные выборки и оставляет модель с наибольшим числом инлайеров.

Напишите `ransac_line(points, threshold, p=0.99)` — RANSAC для прямой.

Скелет — пять шагов:

1. начальное число итераций `N = 1000`, счётчик `k = 0`;
2. пока `k < N`: случайная выборка из 2 точек, модель прямой через них;
3. расстояние всех точек до прямой, инлайеры — те, у кого расстояние < `threshold`;
4. если инлайеров больше, чем в лучшей модели — обновить лучшую, пересчитать `N` по формуле `N = ln(1−p) / ln(1−w²) + 1`, где `w` — доля инлайеров;
5. вернуть `(модель, маска инлайеров)`.

> **Подсказка.** Прямая через две точки `(x1, y1)` и `(x2, y2)`: `a = y1 − y2`, `b = x2 − x1`, `c = x1·y2 − x2·y1`. Расстояние от точки `(x0, y0)` до прямой `ax + by + c = 0`: `|a·x0 + b·y0 + c| / √(a² + b²)`.

In [ ]:
def ransac_line(points, threshold, p=0.99):
    """RANSAC для прямой: возвращает (модель (a, b, c), маска инлайеров)."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 12–16 строк: rng; цикл; выборка 2 точек; модель; расстояния; инлайеры; адаптивный стоп; return
    raise NotImplementedError("TODO 4")
    # ───────────────────────────────────────────────────────────


model, inliers = ransac_line(POINTS, threshold=4.0)
a, b, c = model
# Истинная прямая: θ = 35°, ρ = 120 → a·x + b·y + c = 0
# Проверим, что угол прямой близок к 35°
angle = np.rad2deg(np.arctan2(-a, b)) % 180
assert model is not None, "RANSAC должен найти модель"
assert inliers.mean() > 0.5, "доля инлайеров должна быть > 50%% (сейчас %.0f%%)" % (100 * inliers.mean())
assert abs(angle - 35) < 10 or abs(angle - 35 + 180) < 10, "угол прямой должен быть около 35° (сейчас %.0f°)" % angle
print("TODO 4 ✔ · модель: a=%.2f, b=%.2f, c=%.2f · угол %.0f° · инлайеров %.0f%%" % (a, b, c, angle, 100 * inliers.mean()))

### МНК против Хубера против RANSAC на выбросах (слайд 77)

На синтетической прямой с 30% выбросов: МНК даёт ошибку в несколько градусов, Хубер — почти ноль, RANSAC + refit — тоже почти ноль.

In [ ]:
# МНК через все точки (включая выбросы)
coef_ls = np.polyfit(POINTS[:, 0], POINTS[:, 1], 1)
# Хубер через cv2.fitLine
pts_cv = POINTS.astype(np.float32).reshape(-1, 1, 2)
line_hub = cv2.fitLine(pts_cv, cv2.DIST_HUBER, 0, 0.01, 0.01)
# RANSAC + refit по инлайерам (МНК только по инлайерам)
coef_ransac = np.polyfit(POINTS[inliers, 0], POINTS[inliers, 1], 1)

# Истинный угол: 35° → наклон = tan(90° - 35°) = tan(55°) ≈ 1.43
# Но в системе координат изображения угол может быть другим; проверим относительную ошибку МНК против RANSAC
print("МНК:     k = %.3f, b = %.1f" % (coef_ls[0], coef_ls[1]))
print("Хубер:   направление (%.3f, %.3f)" % (line_hub[0], line_hub[1]))
print("RANSAC:  k = %.3f, b = %.1f" % (coef_ransac[0], coef_ransac[1]))
print("Доля инлайеров: %.0f%%" % (100 * inliers.mean()))

---

## Клиника: ДЗ 2 «Сканер документов» (последние 20 минут)

**ДЗ 2 «Сканер документов»** выдаётся на этой неделе. Что из сегодняшнего занятия туда идёт:

| Тема занятия 7 | Где в ДЗ 2 | Правило |
|----------------|------------|---------|
| Canny | поиск границ документа | сначала `cv2.GaussianBlur` с σ ≈ 1, потом `cv2.Canny`; пороги — из гистограммы градиента, не «из туториала» (слайд 31) |
| HoughLinesP | поиск прямых сторон документа | после `cv2.Canny`; настроить `minLineLength` и `maxLineGap`; схлопнуть дубли (слайд 60) |
| RANSAC | робастная подгонка линий | `cv2.fitLine` с `DIST_HUBER` или RANSAC; МНК при 30% выбросов даёт ошибку 4.3°, Хубер — 0.0° (слайд 77) |
| `findContours` + `approxPolyDP` | поиск контура документа | после `cv2.Canny` + `cv2.morphologyEx` (closing); контур с 4 вершинами — кандидат в документ (слайд 32) |
| `uint8` и `np.clip` | все точечные операции | считать во `float32`, обратно через `np.clip` (слайд 49 из L6) |

**Чек-лист перед сдачей ДЗ 2:**

- `pytest tests -q` — тесты на синтетике;
- на кадрах курса: `cv2.Canny` находит 4 стороны документа, `cv2.HoughLinesP` находит отрезки, `cv2.findContours` возвращает контур с 4 вершинами;
- `uint8`: все арифметические операции — во `float32`, обратно через `np.clip`.

**Чего не показываем:** решение ДЗ целиком и код функций из раздела «своя реализация».

### `findContours` + `approxPolyDP`: поиск контура документа (слайд 32)

In [ ]:
# На кадре с лестницей: найдём контуры и аппроксимируем до многоугольника
edges = cv2.Canny(cv2.GaussianBlur(crop, (0, 0), 1.0), 30, 100)
closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print("найдено контуров: %d" % len(contours))
# Аппроксимация до многоугольника для первых 5 контуров по площади
contours_sorted = sorted(contours, key=cv2.contourArea, reverse=True)[:5]
for i, cnt in enumerate(contours_sorted):
    eps = 0.02 * cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, eps, True)
    print("контур %d: площадь %d, вершин %d" % (i, cv2.contourArea(cnt), len(approx)))

### Вопросы клиники

Принесли вопрос по ДЗ 1 или ДЗ 2 — задавайте сейчас. Если вопросов нет, разбираем связь сегодняшнего занятия с ДЗ 2 «Сканер документов». С занятия 5 ассистент принимает **показ ДЗ**: одно ДЗ за семестр, две минуты — запуск команды и объяснение одного места кода.

---

## ⭐ Звёздочка (по желанию, +0.5, до конца следующей недели)

**Полный конвейер поиска документа** (слайд 5): `Canny` → `findContours` → `approxPolyDP` → четыре угла.

Напишите `find_document_corners(img)` — возвращает массив `(4, 2)` координат четырёх углов контура документа на кадре.

Скелет:

1. Гауссово сглаживание + `cv2.Canny`;
2. морфологическое закрытие (`cv2.morphologyEx` с `MORPH_CLOSE`);
3. `cv2.findContours` + сортировка по площади;
4. `cv2.approxPolyDP` — контур с 4 вершинами.

Ячейка ниже проверит результат на синтетическом «листе на столе».

In [ ]:
def find_document_corners(img):
    """Поиск 4 углов контура документа: Canny → findContours → approxPolyDP."""
    raise NotImplementedError("⭐ звёздочка: полный конвейер поиска документа")


# Синтетический «лист на столе» для проверки звезды (слайд 5)
def make_synthetic_document():
    """Синтетический лист на фоне: 4 угла известны точно."""
    img = np.full((400, 500), 80, np.uint8)  # фон — тёмный стол
    pts = np.array([[100, 80], [380, 90], [370, 320], [110, 310]], np.int32)  # лист
    cv2.fillConvexPoly(img, pts, 220)  # лист — светлый
    return img, pts


try:
    doc_img, doc_pts = make_synthetic_document()
    corners = find_document_corners(doc_img)
except NotImplementedError:
    print("⭐ не заполнена — это необязательная часть; замеры появятся, когда функция будет написана")
else:
    assert corners.shape == (4, 2), "должно быть ровно 4 угла, сейчас %s" % (corners.shape,)
    print("⭐ найдено 4 угла:", corners.tolist())

## Зачёт за занятие

Ассистент ставит зачёт, если:

- [ ] ячейка проверки окружения прошла;
- [ ] **TODO 1–4 выполнены**, все ассерты проходят;
- [ ] вы можете объяснить, **зачем сглаживание перед производной** и как работает **гистерезис в Canny**;
- [ ] на вопросы «почему Собель сглаживает поперёк производной», «как работает аккумулятор Хафа» и «сколько итераций нужно RANSAC при доле выбросов ε = 0.5» есть внятный ответ.

2 % итоговой оценки, ставится в классе. Досылать ничего не нужно. **Занятие 8 — точки интереса: детекторы Харриса и Ши-Томаси.**